[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiaosuhu/Pytorch_1hour_intro/blob/main/pytorch_intro_workshop.ipynb)

# PyTorch for Researchers — Hands-On Notebook
**MIDAS Summer Academy · PyTorch Intro Workshop**

Run each cell top to bottom. Sections match the slide deck 1:1.

- Part 1 — Autograd
- Part 2 — Tensors & GPU
- Part 3 — Loading our EEG spectrogram data
- Part 4 — Dataset & DataLoader
- Part 5 — Training loop anatomy (mechanics demo)
- Part 6 — 🤖 AI-Assisted Coding: generate the real model with Claude
- Part 7 — Put it together: full training run


In [ ]:
import torch
import torch.nn as nn
import numpy as np

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


## Part 1 — Autograd: PyTorch computes derivatives for you

No manual calculus. PyTorch tracks every operation on a tensor and can compute
gradients automatically via `.backward()`.


In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2                    # y = 9.0

y.backward()                  # PyTorch computes dy/dx automatically
print(x.grad)                 # tensor(6.0)  ->  dy/dx = 2x = 2(3) = 6


## Part 2 — Tensors & GPU

"A Tensor is just a numpy array that can live on GPU and track its own gradient."

- **Shape matters more than values** — always check `x.shape` first when debugging
- **`.detach().cpu().numpy()`** — strip grad tracking → off GPU → numpy array. Use `.item()` for single-value scalars (e.g. loss)
- **dtype is your friend** — float32 for models, long for class labels


In [ ]:
# Create tensors
x = torch.tensor([1.0, 2.0, 3.0])
x = torch.randn(32, 128)   # batch of 32

# GPU: one line to move
device = 'cuda' if torch.cuda.is_available() else 'cpu'
x = x.to(device)
print("Running on:", device)

# Track gradients
x = torch.randn(3, requires_grad=True)
print(x)


## Part 3 — Loading Our EEG Spectrogram Data

Synthetic data — same shape/structure as the real HMS 2D CNN pipeline (4-channel
bipolar montage spectrograms, 6-class classification), but **not** real competition
data, so it's safe to share outside Kaggle.

> Replace the URL below with your GitHub raw link once the file is pushed to the repo.


In [ ]:
# --- Download synthetic dataset ---
!wget -q -O pytorch_demo_spectrograms.npz https://raw.githubusercontent.com/xiaosuhu/Pytorch_1hour_intro/main/data/pytorch_demo_spectrograms.npz

data = np.load('pytorch_demo_spectrograms.npz')
X_train, y_train = data['X_train'].astype(np.float32), data['y_train']
X_val, y_val = data['X_val'].astype(np.float32), data['y_val']
label_names = data['label_names']

print("X_train shape:", X_train.shape)   # (150, 4, 32, 64)
print("y_train shape:", y_train.shape)
print("Labels:", list(label_names))


In [ ]:
# Visualize one example per class (channel 0 only, out of the 4 bipolar channels)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 6, figsize=(18, 3))
for label in range(6):
    idx = np.where(y_train == label)[0][0]      # first example of this class
    axes[label].imshow(X_train[idx, 0], aspect='auto', origin='lower', cmap='viridis')
    axes[label].set_title(label_names[label])
    axes[label].set_xlabel('time')
    if label == 0:
        axes[label].set_ylabel('frequency')
plt.tight_layout()
plt.show()


## Part 4 — Dataset & DataLoader

Only 2 methods are required: `__len__` and `__getitem__`.
This is where you put your data loading, augmentation, and normalization logic.


In [ ]:
from torch.utils.data import Dataset, DataLoader

class SpectrogramDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):                # required
        return len(self.X)

    def __getitem__(self, i):         # required
        x = torch.tensor(self.X[i])
        y = torch.tensor(self.y[i], dtype=torch.long)
        return x, y

train_dataset = SpectrogramDataset(X_train, y_train)
val_dataset = SpectrogramDataset(X_val, y_val)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
)

# Sanity check
xb, yb = next(iter(train_loader))
print("Batch x shape:", xb.shape)   # (16, 4, 32, 64)
print("Batch y shape:", yb.shape)


## Part 5 — Training Loop Anatomy

Before we bring in the real architecture (next section), let's see the training
loop mechanics with a **throwaway dummy model** — just a single linear layer on
flattened input. The point here is the *pattern*, not the accuracy.

**3 lines researchers always forget:**
1. `model.train()` before training
2. `optimizer.zero_grad()` before each batch
3. `model.eval()` + `torch.no_grad()` before validation


In [ ]:
# Throwaway dummy model just to see the loop mechanics
dummy_model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(4 * 32 * 64, 6)
).to(device)

optimizer = torch.optim.AdamW(dummy_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

dummy_model.train()                       # <- never forget this
for inputs, targets in train_loader:
    inputs = inputs.to(device)
    targets = targets.to(device)

    optimizer.zero_grad()                 # <- before every batch
    outputs = dummy_model(inputs)
    loss = criterion(outputs, targets)
    loss.backward()
    optimizer.step()                      # <- update weights

print("One dummy pass complete. Loss on last batch:", loss.item())


## Part 6 — 🤖 AI-Assisted Coding: Generate the Real Model

**Workflow: Generate → Audit → Modify**

1. **Generate**: paste the prompt below into Claude (chat) and get a `SpectrogramCNN` class back
2. **Audit**: read every line — ask "why is this here?" before running anything
3. **Modify**: change one thing (e.g. number of filters, `batch_size`), predict the effect, verify

---
**Prompt to paste into Claude(or any AI chatbot you are working with):**

> I'm building a PyTorch classifier for EEG spectrogram data.
>
> Data format:
> - Input: 4-channel 2D spectrogram-like tensors, shape (4, 32, 64) — 4 channels (bipolar EEG montage regions: LL, RL, LP, RP), 32 frequency bins, 64 time steps
> - Output: 6-class classification (Seizure, LPD, GPD, LRDA, GRDA, Other)
>
> Please write:
> 1. A PyTorch `nn.Module` class called `SpectrogramCNN` that takes input of shape `(batch, 4, 32, 64)` and outputs 6 class logits. Use 2-3 `Conv2d` layers with batchnorm and ReLU, followed by adaptive average pooling and a linear classifier head.
> 2. Code to instantiate the model, an `AdamW` optimizer (lr=1e-3), and `CrossEntropyLoss`.
> 3. Keep it simple and well-commented — this is for a live teaching demo, not production code.
>
> Please output only the code, ready to paste into a notebook cell.

---

Paste the generated code into the cell below, replacing the placeholder.


In [ ]:
# 👉 Claude-generated model (Generate -> Audit -> Modify)
#
# Key difference from the placeholder: this keeps spatial structure through
# THREE conv blocks (each halving H and W via stride) before pooling, so the
# network can learn *where* energy shows up in the freq/time grid -- not just
# how much total energy there is.

class SpectrogramCNN(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: (4, 32, 64) -> (16, 16, 32)
            nn.Conv2d(4, 16, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),

            # Block 2: (16, 16, 32) -> (32, 8, 16)
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            # Block 3: (32, 8, 16) -> (64, 4, 8)
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
        )
        # Pool only AFTER spatial patterns have been extracted
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = x.flatten(1)
        return self.classifier(x)


model = SpectrogramCNN().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
print(model)


## Part 7 — Put It Together: Full Training Run

Same loop pattern as Part 5, now with the real model, plus validation.
1 epoch = one full pass over every batch in `train_loader`.

**What to expect vs. the Part 5 placeholder:** the placeholder pools globally
after a single conv layer, so it can only learn "how much total energy" — it
gets stuck around val_acc ≈ 0.33 (2 of 6 classes). This model keeps spatial
structure through 3 conv blocks before pooling, so it should be able to tell
classes apart by *where* the energy sits in the freq/time grid, not just how
much there is. Expect val_acc to climb well past 0.33 within these 10 epochs.


In [ ]:
def evaluate(model, loader):
    model.eval()                          # <- switch mode
    total, correct, loss_sum = 0, 0, 0.0
    with torch.no_grad():                 # <- saves memory
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss_sum += criterion(outputs, targets).item() * inputs.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == targets).sum().item()
            total += inputs.size(0)
    return loss_sum / total, correct / total


num_epochs = 10   # small dataset + workshop time budget -> keep short

for epoch in range(num_epochs):
    model.train()                         # <- never forget this
    running_loss = 0.0
    for inputs, targets in train_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()             # <- before every batch
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()                  # <- update weights
        running_loss += loss.item() * inputs.size(0)

    train_loss = running_loss / len(train_dataset)
    val_loss, val_acc = evaluate(model, val_loader)
    print(f"Epoch {epoch+1:2d} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f} | val_acc {val_acc:.3f}")


## Next Steps

**Tonight's challenge:** Run the HMS EEG 1D-CNN baseline on Kaggle. Change one
hyperparameter. Watch the loss curve. Ask Claude Code one question about the output.

**Resources**
- Official docs → pytorch.org/docs/stable/index.html
- HMS EEG Kaggle → kaggle.com/competitions/hms-harmful-brain-activity-classification
- Prompt engineering → docs.claude.com/en/docs/build-with-claude/prompt-engineering/overview
- Workshop repo → github.com/xiaosuhu/hms-eeg-review-paper
